In [4]:
%useLatestDescriptors
%use dataframe(1.0.0-Beta5n)
%use kandy
%use serialization

In [ ]:
@file:Repository("https://repo.gradle.org/gradle/libs-releases") //
@file:DependsOn("org.gradle:gradle-tooling-api:9.6.1")

import org.gradle.tooling.GradleConnector
import java.io.File

GradleConnector.newConnector().forProjectDirectory(File("./../../")).connect().use { connection ->
    connection.newBuild()
            .forTasks(
                "clean",
                "reverseBytesBenchmark", "reverseBitsBenchmark",
                "singleBitsBenchmark", "multipleBitsBenchmark"
            )
            .setStandardOutput(System.out)
            .setStandardError(System.err)
            .run()
}

In [5]:
@file:OptIn(ExperimentalSerializationApi::class)

import kotlinx.serialization.ExperimentalSerializationApi
import kotlinx.serialization.SerialName
import kotlinx.serialization.Serializable
import kotlinx.serialization.json.Json
import kotlinx.serialization.json.decodeFromStream
import org.jetbrains.kotlinx.kandy.dsl.plot
import org.jetbrains.kotlinx.kandy.letsplot.layers.bars
import java.nio.file.Path
import kotlin.io.path.PathWalkOption
import kotlin.io.path.extension
import kotlin.io.path.inputStream
import kotlin.io.path.name
import kotlin.io.path.nameWithoutExtension
import kotlin.io.path.walk

fun String.suffixSimilarity(other: String): Double {
    val a = this.reversed()
    val b = other.reversed()
    val maxLen = maxOf(a.length, b.length)
    if (maxLen == 0) return 1.0
    var match = 0
    for (i in 0 until minOf(a.length, b.length)) {
        if (a[i] == b[i]) match++ else break
    }
    return match.toDouble() / maxLen
}

inline fun <T> List<T>.groupBySimilarityChain(crossinline selector: (T) -> String): List<T> {
    if (isEmpty()) return emptyList()
    val unused = toMutableSet()
    val result = ArrayList<T>()
    var current = unused.first()
    result += current
    unused -= current
    while (unused.isNotEmpty()) {
        val next = unused.maxBy { selector(it).suffixSimilarity(selector(current)) }
        result += next
        unused -= next
        current = next
    }
    return result
}

@Serializable
data class PrimaryMetric(
    val score: Double,
    val scoreError: Double,
    val scoreUnit: String
)

@Serializable
data class Benchmark(
    @SerialName("benchmark") val name: String,
    val warmupIterations: Int,
    val measurementIterations: Int,
    val primaryMetric: PrimaryMetric
) {
    inline val cleanName: String
        get() = name.substringBeforeLast('.').substringAfterLast('.')
}

val json: Json = Json { ignoreUnknownKeys = true }

Path.of("./../build/reports/benchmarks")
        .walk()
        .filter { filePath -> filePath.extension == "json" }
        .map { filePath ->
            filePath.inputStream().use { stream ->
                filePath to json.decodeFromStream<List<Benchmark>>(stream)
            }
        }.map { (filePath, report) ->
            val sortedReport = report.groupBySimilarityChain(Benchmark::name)
            val plotName = filePath.parent.parent.name
            val platform = filePath.nameWithoutExtension
            val df = dataFrameOf(
                "name" to sortedReport.map(Benchmark::cleanName),
                "score" to sortedReport.map { benchmark -> benchmark.primaryMetric.score },
                "score_min" to sortedReport.map { benchmark -> benchmark.primaryMetric.score - benchmark.primaryMetric.scoreError },
                "score_max" to sortedReport.map { benchmark -> benchmark.primaryMetric.score + benchmark.primaryMetric.scoreError }
            )
            val plot = plot(df) {
                layout {
                    size = 1200 to 800
                    title = "$plotName ($platform)" // Parent of parent is suite name
                    xAxisLabel = "Implementation"
                    yAxisLabel = if("reverse" in plotName.lowercase()) "ops/s" else "MiB/s"
                    theme = Theme.DARCULA
                }
                bars {
                    x("name")
                    y("score") {
                        axis {
                            breaks(format = ".2f")
                        }
                    }
                    fillColor("name")
                }
                errorBars {
                    x("name")
                    yMin("score_min")
                    yMax("score_max")
                }
            }
            plot.save("./../../docs/${plotName}_$platform.png")
            plot
        }.forEach(::DISPLAY)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="IvuJtN" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("IvuJtN");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"singleBits (wasmJs)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[39.517017694473836,6.427294879414541,46.5025874253566,13.725313293625891],
"name":["CommonReadSingleBitsBenchmark","KarbideReadSingleBitsBenchmark","CommonWriteSingleBitsBenchmark","KarbideWriteSingleBitsBenchmark"],
"score_max":[40.78771699744925,6.520163016788227,46.85244876166893,14.028270016763235],
"score_min":[38.24631839149842,6.334426742040855,46.15272608904427,13.422356570488548]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"19"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 15.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 25.00 
 
 
 
 
 
 
 30.00 
 
 
 
 
 
 
 35.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 45.00 
 
 
 
 
 
 
 
 
 singleBits (wasmJs) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Commo

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="yKzv1a" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("yKzv1a");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"singleBits (linuxX64)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[29.6863403240988,9.45141516711403,54.75304984180142,29.200039355928112],
"name":["CommonReadSingleBitsBenchmark","KarbideReadSingleBitsBenchmark","CommonWriteSingleBitsBenchmark","KarbideWriteSingleBitsBenchmark"],
"score_max":[30.194284738992888,9.78157678775117,55.99510385786791,29.7133874670097],
"score_min":[29.178395909204713,9.12125354647689,53.51099582573493,28.686691244846525]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"23"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 15.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 25.00 
 
 
 
 
 
 
 30.00 
 
 
 
 
 
 
 35.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 45.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 55.00 
 
 
 
 
 
 
 
 
 singleBits (linuxX64) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 KarbideReadSingleB

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="TloKfR" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("TloKfR");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"singleBits (wasmWasi)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[40.00016160793281,6.240644978064967,45.28990907545394,12.525930630625634],
"name":["CommonReadSingleBitsBenchmark","KarbideReadSingleBitsBenchmark","CommonWriteSingleBitsBenchmark","KarbideWriteSingleBitsBenchmark"],
"score_max":[40.65503879777796,6.31445156553495,46.14788878719508,12.668779305548826],
"score_min":[39.34528441808766,6.166838390594985,44.431929363712804,12.383081955702442]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"27"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 15.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 25.00 
 
 
 
 
 
 
 30.00 
 
 
 
 
 
 
 35.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 45.00 
 
 
 
 
 
 
 
 
 singleBits (wasmWasi) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 C

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="fH0T2P" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("fH0T2P");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"singleBits (js)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[3.108733942137108,1.1121507700546966,14.626819155539062,1.9769344897486865],
"name":["CommonReadSingleBitsBenchmark","KarbideReadSingleBitsBenchmark","CommonWriteSingleBitsBenchmark","KarbideWriteSingleBitsBenchmark"],
"score_max":[3.1600080790433074,1.14783938915923,15.503887451331833,2.0026418047715744],
"score_min":[3.0574598052309083,1.0764621509501633,13.74975085974629,1.9512271747257983]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"31"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 1.00 
 
 
 
 
 
 
 2.00 
 
 
 
 
 
 
 3.00 
 
 
 
 
 
 
 4.00 
 
 
 
 
 
 
 5.00 
 
 
 
 
 
 
 6.00 
 
 
 
 
 
 
 7.00 
 
 
 
 
 
 
 8.00 
 
 
 
 
 
 
 9.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 11.00 
 
 
 
 
 
 
 12.00 
 
 
 
 
 
 
 13.00 
 
 
 
 
 
 
 14.00 
 
 
 
 
 
 
 15.00 
 
 
 
 
 
 
 16.00 
 
 
 
 
 
 
 
 
 singleBits (js) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="R3zNaz" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("R3zNaz");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"singleBits (jvm)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[208.9748234054769,24.301026727046256,59.042272274083416,40.65413835103021],
"name":["CommonReadSingleBitsBenchmark","KarbideReadSingleBitsBenchmark","CommonWriteSingleBitsBenchmark","KarbideWriteSingleBitsBenchmark"],
"score_max":[221.82133580194267,26.380050990519205,61.27671426872986,41.530785665940144],
"score_min":[196.12831100901116,22.222002463573308,56.80783027943697,39.77749103612028]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"35"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 CommonWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 KarbideWriteSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 60.00 
 
 
 
 
 
 
 80.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 120.00 
 
 
 
 
 
 
 140.00 
 
 
 
 
 
 
 160.00 
 
 
 
 
 
 
 180.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 220.00 
 
 
 
 
 
 
 
 
 singleBits (jvm) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CommonReadSingleBitsBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 KarbideReadS